# Notebook 5: Add Emotion Labels to Kaiaulu Output

### Prerequisites

Before running this notebook for a given project:

1. **Notebook 2 must have been run**: `jira_comment_emotion` must exist in PostgreSQL.
2. **`download_jira_issues.Rmd` must have been run** for the project using the "Issues by Key Range" downloader. The JSON output must be saved to the `issue_comments/` folder specified in the project's Kaiaulu config.
3. **PostgreSQL must be running locally**

### Step 1: Import dependencies

In [ ]:
import os
import json
import glob
import pandas as pd
from sqlalchemy import create_engine, text

### Step 2: Set configuration

Set `PROJECT_KEY` to the Jira project key you downloaded. Update the paths (`ISSUE_COMMENTS_PATH` and `OUTPUT_PATH`) and connection details to match your local setup

In [ ]:
# Project to process
PROJECT_KEY = "ADD_PROJECT_KEY_HERE"

# Path to the issue_comments folder for this project
# This should match the issue_comments path in the project's Kaiaulu config
ISSUE_COMMENTS_PATH = os.path.expanduser(
    f"~/PATH_TO/rawdata/jira/{PROJECT_KEY.lower()}/issue_comments/"
)

# Where to save the joined output CSV
OUTPUT_PATH = os.path.expanduser(
    f"~/PATH_TO/rawdata/jira/{PROJECT_KEY.lower()}/{PROJECT_KEY.lower()}_emotion_comments_joined.csv"
)

# PostgreSQL connection details
PG_HOST     = "localhost"
PG_PORT     = 5432
PG_USER     = "postgres"
PG_PASSWORD = "ADD_YOUR_PASSWORD_HERE"
PG_DB       = "jira"

engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}")

print(f"Project:              {PROJECT_KEY}")
print(f"Issue comments path:  {ISSUE_COMMENTS_PATH}")
print(f"Output CSV:           {OUTPUT_PATH}")

### Step 3: Load emotion labels for this project from PostgreSQL

Pull the emotion scores for this project out of PostgreSQL, along with the issue key and timestamp for each comment.

In [ ]:
with engine.connect() as con:
    emotion_labels = pd.read_sql(text("""
        SELECT
            e.comment_id    AS comment_id_db,
            r.key           AS issue_key,
            c.creationdate  AS db_timestamp,
            e.group_number,
            e.love,
            e.joy,
            e.sadness,
            e.anger,
            e.surprise,
            e.fear
        FROM jira_comment_emotion e
        INNER JOIN jira_issue_comment c ON e.comment_id = c.id
        INNER JOIN jira_issue_report r  ON c.issue_report_id = r.id
        WHERE SPLIT_PART(r.key, '-', 1) = :project_key;
    """), con, params={'project_key': PROJECT_KEY})

print(f"Emotion labels loaded for {PROJECT_KEY}: {len(emotion_labels)} rows")
display(emotion_labels.head())

### Step 4: Parse the Kaiaulu downloaded JSON files

Parse the JSON files Kaiaulu downloaded and return a table of comments with their text, author, and timestamps.

In [ ]:
json_files = glob.glob(os.path.join(ISSUE_COMMENTS_PATH, '*.json'))
print(f"JSON files found: {len(json_files)}")


rows = []
for json_path in json_files:
   with open(json_path) as f:
       data = json.load(f)
   for issue in data.get('issues', []):
       issue_key = issue.get('key')
       comments  = issue.get('fields', {}).get('comment', {}).get('comments', [])
       for c in comments:
           author = c.get('author', {})
           rows.append({
               'comment_id':    int(c['id']),
               'issue_key':     issue_key,
               'body':          c.get('body'),
               'author_login':  author.get('name'),
               'author_name':   author.get('displayName'),
               'timestamp':     c.get('created'),
               'updated_at':    c.get('updated'),
           })


kaiaulu_comments = pd.DataFrame(rows)


print(f"Total comments parsed: {len(kaiaulu_comments)}")
print(f"Unique issues:         {kaiaulu_comments['issue_key'].nunique()}")
display(kaiaulu_comments.head())

### Step 5: INNER JOIN - emotion labels to Kaiaulu comments

Join the PostgreSQL emotion labels against Kaiaulu's Jira issue comments on `timestamp` and `issue_key`.

In [ ]:
emotion_labels['timestamp_utc'] = (
    pd.to_datetime(emotion_labels['db_timestamp'])
    .dt.tz_localize('Etc/GMT-8')   # DB stored in UTC+8
    .dt.tz_convert('UTC')
)

kaiaulu_comments['timestamp_utc'] = pd.to_datetime(kaiaulu_comments['timestamp'],  utc=True)

joined = kaiaulu_comments.merge(
    emotion_labels,
    on=['issue_key', 'timestamp_utc'],
    how='inner'
)

# Drop the temporary normalized column used only for merging
joined = joined.drop(columns=['timestamp_utc', 'db_timestamp'])

# Reorder columns
col_order = [
    'comment_id', 'comment_id_db', 'issue_key', 'group_number',
    'body', 'timestamp', 'updated_at',
    'author_login', 'author_name',
    'love', 'joy', 'sadness', 'anger', 'surprise', 'fear'
]
joined = joined[col_order]

print(f"Rows in Kaiaulu comments:      {len(kaiaulu_comments)}")
print(f"Rows in emotion labels:        {len(emotion_labels)}")
print(f"Rows after INNER JOIN:         {len(joined)}")
print(f"Unique issues matched:         {joined['issue_key'].nunique()}")
print()
display(joined.head(10))


### Step 6: Write output CSV

Save the joined dataset as a CSV file to the project's rawdata folder.

In [ ]:
joined.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(joined)} rows to:")
print(f"  {OUTPUT_PATH}")

### You're done!

The output CSV contains every Kaiaulu-downloaded comment for this project that has a matching emotion label. 

Here are the columns within the contextualized CSV:

| Column | Description |
|---|---|
| `comment_id` | Jira comment ID from the live API (via Kaiaulu) |
| `comment_id_db` | Jira comment ID from the 2016 SQL dump (may differ from live API) |
| `issue_key` | e.g. `HARMONY-1065` |
| `group_number` | which rater group labeled this comment |
| `body` | comment text |
| `timestamp` | comment creation date |
| `updated_at` | comment last updated |
| `author_login` | Jira username |
| `author_name` | display name |
| `love`, `joy`, `sadness`, `anger`, `surprise`, `fear` | inter-rater agreement scores (0–1) |

**To run for a different project:** Change `PROJECT_KEY` in Step 2, make sure you have run `download_jira_issues.Rmd` for that project, and re-run the notebook from Step 2 onwards.